# 03 — Baseline Modeling
## Credit Risk: Loss Given Default (LGD) Prediction

**Objectives:**
- Establish the segment-average LGD benchmark (competition baseline)
- Train and evaluate Ridge regression (linear baseline)
- Train and evaluate LightGBM (strong non-parametric baseline)
- Compare all models against benchmark: MAE/RMSE/R²
- Log runs to MLflow

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config
from src.models.baseline import run_baselines, compute_segment_average_benchmark, compute_metrics

config = load_config()
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete')

## 1. Load Feature Matrix

In [ ]:
features_path = Path(config.data.processed_dir) / 'features.parquet'
if features_path.exists():
    df = pd.read_parquet(features_path)
    print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
    print('Target:', config.features.target)
    print('Features:', config.features.categorical + config.features.numeric)
else:
    print('Run feature engineering first: python src/data/features.py')

## 2. Run All Baselines

In [ ]:
# TODO: Run baseline models
# This logs to MLflow automatically

results = run_baselines(config)

print('\n=== Baseline Results ===')
for model_name, metrics in results.items():
    print(f'\n{model_name}:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')

## 3. Benchmark Comparison Table

In [ ]:
# TODO: Format results as comparison table

comparison_data = []
benchmark_test_mae = results.get('segment_average_benchmark', {}).get('test_mae', None)
benchmark_test_rmse = results.get('segment_average_benchmark', {}).get('test_rmse', None)

for model_name, metrics in results.items():
    row = {
        'Model': model_name,
        'Test MAE': metrics.get('test_mae', None),
        'Test RMSE': metrics.get('test_rmse', None),
        'Test R²': metrics.get('test_r2', None),
    }
    if benchmark_test_mae and model_name != 'segment_average_benchmark' and row['Test MAE']:
        row['MAE Improvement vs Benchmark'] = f"{(benchmark_test_mae - row['Test MAE']) / benchmark_test_mae * 100:.1f}%"
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

## 4. Metric Visualization

In [ ]:
# TODO: Bar chart comparing models on test MAE
models = [r['Model'] for r in comparison_data]
mae_values = [r['Test MAE'] for r in comparison_data if r['Test MAE'] is not None]

if mae_values:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#6B7280'] + ['#2563EB'] * (len(mae_values) - 1)  # Grey for benchmark
    ax.bar(models[:len(mae_values)], mae_values, color=colors, alpha=0.85)
    ax.set_ylabel('Test MAE')
    ax.set_title('Model Comparison: Test MAE (Lower is Better)')
    ax.grid(True, axis='y', alpha=0.3)
    if benchmark_test_mae:
        ax.axhline(benchmark_test_mae, color='red', linestyle='--', alpha=0.7, label='Benchmark')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. LightGBM Feature Importance

In [ ]:
# TODO: LightGBM native feature importance
# (Run this cell after train_lightgbm returns the model object)
# lgb_model = ...  # from train_lightgbm()
# importance = lgb_model.feature_importance(importance_type='gain')
# feature_names = config.features.categorical + config.features.numeric
# ...
print('LightGBM feature importance: run run_baselines() and retrieve the lgb_model object')

## 6. MLflow Run Summary

In [ ]:
# TODO: List MLflow runs for the baseline experiment
try:
    import mlflow
    mlflow.set_tracking_uri(config.mlflow.tracking_uri)
    runs = mlflow.search_runs(experiment_names=[config.mlflow.experiment_name])
    if not runs.empty:
        print('MLflow runs:')
        display_cols = ['run_id', 'tags.mlflow.runName', 'metrics.test_mae', 'metrics.test_r2', 'start_time']
        display_cols = [c for c in display_cols if c in runs.columns]
        print(runs[display_cols].to_string(index=False))
except ImportError:
    print('MLflow not installed')

## Key Findings

*To be completed after running:*

1. **Segment-average benchmark MAE:** [value]
2. **Ridge MAE vs benchmark:** [value and % improvement]
3. **LightGBM MAE vs benchmark:** [value and % improvement]
4. **Does LightGBM meet the ≥15% MAE improvement target?** [Yes/No]
5. **Top LightGBM features by gain importance:** [list]
6. **Decision:** [Whether PyTorch model is needed or LightGBM is sufficient]